In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
import json
import torch
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from datasets import Dataset
from trl import SFTTrainer, SFTConfig


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [13]:
MAX_SEQ_LENGTH = 2048   # veri setindeki p95 ~510, maks ~638 token (thinking dahil, yaklaşık ölçüm) + tampon
LORA_RANK = 16
LORA_ALPHA = 16
BATCH_SIZE = 1
GRAD_ACCUM = 8          # etkin batch = 1*8 = 8
NUM_EPOCHS = 3
LEARNING_RATE = 2e-4

TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

In [4]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3.5-4B",
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,  # otomatik seçilsin (bf16 destekleniyorsa o, yoksa fp16)
)

==((====))==  Unsloth 2026.7.4: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # False if not finetuning vision layers
    finetune_language_layers   = True, # False if not finetuning language layers
    finetune_attention_modules = True, # False if not finetuning attention layers
    finetune_mlp_modules       = True, # False if not finetuning MLP layers
    r=LORA_RANK,
    target_modules=TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    use_rs_lora=True,
    use_gradient_checkpointing="unsloth",  # VRAM tasarrufu için zorunlu
    random_state=42,
)

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


In [7]:
with open("/content/sondaj_finetune_veriseti.json", "r", encoding="utf-8") as f:
    raw = json.load(f)

In [9]:
dataset = Dataset.from_json("/content/sondaj_finetune_veriseti.json")

Generating train split: 0 examples [00:00, ? examples/s]

In [11]:
dataset

Dataset({
    features: ['content', 'images', 'role', 'thinking', 'tool_calls'],
    num_rows: 3822
})

In [16]:
def format_chat_data(batch):
    texts = []
    # Verideki tüm elemanları ikişerli (user ve assistant) alıyoruz
    for i in range(0, len(batch["role"]) - 1, 2):
        if batch["role"][i] == "user" and batch["role"][i+1] == "assistant":
            user_content = batch["content"][i]
            asst_content = batch["content"][i+1]

            messages = [
                {"role": "user", "content": user_content},
                {"role": "assistant", "content": asst_content}
            ]

            # Tokenizer'ın sohbet şablonunu uyguluyoruz
            text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False
            )
            texts.append(text)

    return {"text": texts}

# Veri setini dönüştürüyoruz
formatted_dataset = dataset.map(format_chat_data, batched=True, remove_columns=dataset.column_names)

Map:   0%|          | 0/3822 [00:00<?, ? examples/s]

In [21]:
# from transformers import TrainingArguments
from unsloth import UnslothTrainer, UnslothTrainingArguments

trainer = UnslothTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted_dataset,
    eval_dataset = None,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,


    args = UnslothTrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        num_train_epochs = 3,

        # Use warmup_ratio and num_train_epochs for longer runs!
        max_steps = 10,
        warmup_steps = 2,
        # warmup_ratio = 0.1,
        # num_train_epochs = 1,

        # Select a 2 to 10x smaller learning rate for the embedding matrices!
        learning_rate = 5e-5,
        embedding_learning_rate = 1e-6,

        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "/content/",
        report_to = "none", # Use this for WandB etc
    ),
)

Unsloth: Switching to float32 training since model cannot work with float16


In [25]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,911 | Num Epochs = 1 | Total steps = 10
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 21,233,664 of 4,560,499,200 (0.47% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,1.434383
2,1.596993
3,1.915522
4,1.200343
5,1.407447
6,1.700724
7,1.617863
8,1.494111
9,1.538075
10,1.156784


In [27]:
model.push_to_hub_merged("uzcaliskan/magibu-llm-fine_tuned_sondaj",
                         tokenizer, token = "")

Unsloth: Restored added_tokens_decoder metadata in uzcaliskan/magibu-llm-fine_tuned_sondaj/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0002-of-00002.safetensors:   1%|          | 24.0MB / 3.99GB            



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [05:26<00:00, 163.32s/it]


Unsloth: Merge process complete. Saved to `/content/uzcaliskan/magibu-llm-fine_tuned_sondaj`


In [28]:
!ollama serve

/bin/bash: line 1: ollama: command not found


In [29]:
dataset.push_to_hub("uzcaliskan/magibu_dataset_drilling", )

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########|  248kB /  248kB            

CommitInfo(commit_url='https://huggingface.co/datasets/uzcaliskan/magibu_dataset_drilling/commit/44b60ad3c0589b1d8272f64175681b5163bccea5', commit_message='Upload dataset', commit_description='', oid='44b60ad3c0589b1d8272f64175681b5163bccea5', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/uzcaliskan/magibu_dataset_drilling', endpoint='https://huggingface.co', repo_type='dataset', repo_id='uzcaliskan/magibu_dataset_drilling'), pr_revision=None, pr_num=None)